## Text Extraction, Cleaning, and Rephrasing from multiple books

In [1]:
import pandas as pd
import traceback
import json
from PyPDF2 import PdfReader
import os

from bson import ObjectId
from motor.motor_asyncio import AsyncIOMotorClient
from langchain_openai import ChatOpenAI
from langchain import PromptTemplate, LLMChain

from image_que_prompts import (
EXTRACTION_SYSTEM,
EXTRACTION_USER,
EXTRACTION_OUTPUT_FORMAT,
EMPTY_EXTRACTION_OUTPUT_FORMAT,
CLEAN_REPHRASE_USER,
CLEAN_REPHRASE_SYSTEM,
REPHRASE_OUTPUT_FORMAT,
REPHRASE_OUTPUT_EXAMPLE,
BIOLOGY_SOLUTION_SYSTEM_PROMPT,
CHEMISTRY_SOLUTION_SYSTEM_PROMPT,
MATHEMATICS_SOLUTION_SYSTEM_PROMPT,
PHYSICS_SOLUTION_SYSTEM_PROMPT,
DISTRACTOR_SYSTEM_PROMPT
)

from config import (
    OPENAI_API_KEY,
    MONGO_URL,
    DB_NAME
)

e:\Questions_extraction\cbse\image_que_prompts.py:1: SyntaxWarning: invalid escape sequence '\i'
  EXTRACTION_SYSTEM = """


In [2]:
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "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"
# os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
# os.environ["LANGCHAIN_PROJECT"] = "Questions Extractor"

# Pipeline

In [3]:
import json
import re

def parse_Response(response):
    """
    Parse a JSON object from a response string or return it if it's already a dict.

    This function attempts to extract a valid JSON substring from the response,
    clean it up, and parse it into a Python dictionary. It handles cases such as:
      - Extra content or multiple JSON objects concatenated.
      - Invalid escape sequences in the JSON string.

    Args:
        response (str or dict): The response containing JSON content.

    Returns:
        dict or None: The parsed JSON object, or None if parsing fails.
    """
    # If the response is already a dictionary, return it as is.
    if isinstance(response, dict):
        return response

    # If the response is a string, try to extract and parse the JSON.
    if isinstance(response, str):
        # Use regex to extract the first substring that starts with '{' and ends with '}'
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            json_str = match.group(0)
        else:
            print("No valid JSON content found in the response.")
            return None

        try:
            # Clean up the JSON string by removing newlines and unwanted characters.
            cleaned_json = json_str.replace("\n", "").replace("\(", "").replace("\)", "")
            return json.loads(cleaned_json)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON response: {e.__class__.__name__} - {e}")
            
            # Handle "Extra data" errors (often due to multiple JSON objects concatenated).
            if str(e).startswith("Extra data"):
                # Split on a pattern like '}\n{' which may separate JSON objects.
                json_parts = re.split(r'}\s*\n\s*{', json_str)
                if json_parts:
                    first_json = json_parts[0].strip()
                    if not first_json.endswith('}'):
                        first_json += '}'
                    return parse_Response(first_json)
            
            # Handle invalid escape sequence issues.
            elif "Invalid \\escape" in str(e):
                print("Before handling escape issues:", json_str)
                # Replace problematic escape sequences with temporary placeholders.
                str1 = json_str.replace("\\\\\\\\\\", "fvback")
                str1 = str1.replace("\\\\\\\\", "frback")
                str1 = str1.replace("\\\\\\", "trlback")
                str1 = str1.replace("\\\\", "dblback")
                # Re-escape the backslashes.
                str1 = str1.replace("\\", "\\\\")
                # Restore the placeholders back to the intended escaped sequences.
                str1 = str1.replace("fvback", "\\\\\\\\\\")
                str1 = str1.replace("frback", "\\\\\\\\")
                str1 = str1.replace("trlback", "\\\\\\")
                str1 = str1.replace("dblback", "\\\\")
                strfinal = str1.replace("\n", "")
                print("After handling escape sequence issues:", strfinal)
                try:
                    return json.loads(strfinal)
                except json.JSONDecodeError as inner_e:
                    print("Still failing to parse JSON after handling escapes:", inner_e)
                    return None
            else:
                print("Actual JSON content was:", json_str)
                return None
    else:
        print("Unsupported response type:", type(response))
        return None



<>:35: SyntaxWarning: invalid escape sequence '\('
<>:35: SyntaxWarning: invalid escape sequence '\)'
<>:35: SyntaxWarning: invalid escape sequence '\('
<>:35: SyntaxWarning: invalid escape sequence '\)'
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_24708\4070082387.py:35: SyntaxWarning: invalid escape sequence '\('
  cleaned_json = json_str.replace("\n", "").replace("\(", "").replace("\)", "")
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_24708\4070082387.py:35: SyntaxWarning: invalid escape sequence '\)'
  cleaned_json = json_str.replace("\n", "").replace("\(", "").replace("\)", "")


In [4]:
# All question collection

def collect_questions_from_chapter_with_Langchain(subject, chapter_text, chapter_name, grade):
    try:
        # initialize the ChapOpenAI object
        llm = ChatOpenAI(
            model_name="gpt-4o",
            api_key= OPENAI_API_KEY,
            temperature=0.2,
        )
        # match subject:
        #     case "mathematics":
        #         examples=MATHEMATICS_EXTRACTION_OUTPUT_EXAMPLE
        #     case "biology":
        #         examples=BIOLOGY_EXTRACTION_OUTPUT_EXAMPLE
        #     case "physics":
        #         examples =PHYSICS_EXTRACTION_OUTPUT_EXAMPLE
        #     case "chemistry":
        #         examples=CHEMISTRY_EXTRACTION_OUTPUT_EXAMPLE

        #formatting prompts
        system = EXTRACTION_SYSTEM.format(" ",grade = grade, subject = subject,EMPTY_EXTRACTION_OUTPUT_FORMAT = EMPTY_EXTRACTION_OUTPUT_FORMAT, out_format = EXTRACTION_OUTPUT_FORMAT)
        user = EXTRACTION_USER.format(chapter_name = chapter_name, chapter_text = chapter_text)

        #generating response from api call
        response = llm.invoke(
            [
                ("system", system),
                ("human", user)
            ],
        )
        return response
    except Exception as e:
        print("Error with in extraction: ", type(e).__name__, "–", e, "\n", traceback.format_exc())
        return None

In [5]:
CLEAN_REPHRASE_SYSTEM.format(" ",out_example = REPHRASE_OUTPUT_EXAMPLE, out_format = REPHRASE_OUTPUT_FORMAT)

'\nYou are a helpful academic assistant who converts the valid questions with images which are in LateX format into multiple choice questions with descriptions without options and maps them to a topic from given list \nYou will be given questions with images which are in LateX format.\nImages are the part of the questions . Do not consider them seperate or consider as options. \n\nRULES for mapping ,rephrasing and  LaTeX formating:\n    - Remove the "\\includegraphics" from the questions if included.\n    - Check the given questions are in LateX format and if not convert them into LateX format  . Do not make changes in the image_ids text\n    - Do not change or generate any new questions or image ids text.\n    - Convert all questions into multiple choice questions without options. The image is part of the question and options are not always the images they are the text moslty.\n    - Rephrase the questions as questions should not be exact same as original questions . Do not change the

In [6]:
def clean_rephrase_question(question_text, topic_list):
    try:
        # initialize the ChapOpenAI object
        llm = ChatOpenAI(
            model_name="gpt-4o",
            api_key= OPENAI_API_KEY,
            temperature=0.2,
            # verbose = True
        )

        #formatting prompts
        system = CLEAN_REPHRASE_SYSTEM.format(" ",out_example = REPHRASE_OUTPUT_EXAMPLE, out_format = REPHRASE_OUTPUT_FORMAT)
        user = CLEAN_REPHRASE_USER.format(question_text = question_text, topic_list = topic_list)

        #generating response from api call
        response = llm.invoke(
            [
                ("system", system),
                ("human", user)
            ]
        )
        return response
    except Exception as e:
        print("Error cleaning text: ", type(e).__name__, "–", e, "\n", traceback.format_exc())
        return question_text  # Return original text in case of an error

In [7]:
import os
import traceback
from pathlib import Path
import zipfile
import tempfile
import shutil

def read_data_from_latex(publication: str, chapter_name: str , grade:str) -> str:
    """
    Read LaTeX content from a specified publication and chapter zip file.
    
    Args:
        publication (str): Name of the publication (e.g., 'mtg')
        chapter_name (str): Name of the chapter (e.g., 'Life Processes')
    
    Returns:
        str: Content of the LaTeX file if successful, empty string if failed
    """
    try:
        # Construct the zip file path
        zip_path = os.path.join(r"textbooks", publication, f"{chapter_name}_{grade}.zip")
        
        # Verify zip file exists
        if not os.path.exists(zip_path):
            raise FileNotFoundError(f"Zip file not found: {zip_path}")
        
        # Create a temporary directory to extract files
        with tempfile.TemporaryDirectory() as temp_dir:
            # Extract the zip file
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(temp_dir)
            
            # Find directories starting with 2025
            content_dirs = [d for d in os.listdir(temp_dir) 
                          if os.path.isdir(os.path.join(temp_dir, d)) and 
                          d.startswith('2025')]
            
            if not content_dirs:
                raise FileNotFoundError(f"No content directories found in zip file")
                
            # Get the latest directory
            latest_dir = sorted(content_dirs)[-1]
            dir_path = os.path.join(temp_dir, latest_dir)
            
            # Find the .tex file in the directory
            tex_files = [f for f in os.listdir(dir_path) if f.endswith('.tex')]
                    
            if not tex_files:
                raise FileNotFoundError(f"No .tex file found in {dir_path}")
                
            # Full path to the tex file
            latex_path = os.path.join(dir_path, tex_files[0])
            
            # Read the content
            with open(latex_path, 'r', encoding='utf-8') as file:
                content = file.read()
                
            print(f"Successfully processed LaTeX file from zip: {latex_path}")
            return content

    except Exception as e:
        print(
            "Failed to process the LaTeX file. Exception Occurred:",
            type(e).__name__,
            "–",
            e,
            "\n",
            traceback.format_exc()
        )
        return ""

In [8]:
def extract_questions(subject: str, chapter_texts: list, chapter_name: str, grade: str) -> dict:
    try:
        if not isinstance(chapter_texts, list) or not chapter_texts:
            return {"questions": []}
            
        formatted_output = {
            "questions": []
        }
        chunk_size = 250
        question_counter = 1
        
        for i in range(0, len(chapter_texts), chunk_size):
            chunk = chapter_texts[i:i + chunk_size]
            if not chunk:
                continue
                
            print(f"\nProcessing chunk {i//chunk_size + 1}/{-(-len(chapter_texts)//chunk_size)}")
            
            response = collect_questions_from_chapter_with_Langchain(
                subject=subject,
                chapter_text='\n'.join(chunk),
                chapter_name=chapter_name,
                grade=grade
            )
            
            print("\nDebug - Raw response:", response)
            
            # Handle response based on its type
            if hasattr(response, 'content'):
                try:
                    response_data = parse_Response(response.content)
                except Exception as parse_error:
                    print(f"Error parsing response content: {parse_error}")
                    continue
            else:
                response_data = response
                
            print("\nDebug - Processed response data:", response_data)
            
            if isinstance(response_data, dict) and "questions" in response_data:
                for question_obj in response_data["questions"]:
                    print(f"\nDebug - Processing question: {question_obj}")
                    
                    try:
                        # Extract question data from the new structure
                        if isinstance(question_obj, dict) and "question" in question_obj:
                            question_data = question_obj["question"]
                            
                            formatted_question = {
                                f"question_{question_counter}": {
                                    "question_text": question_data.get("question_text", ""),
                                    "image_ids": [
                                        img_dict for img_dict in question_data.get("image_ids", [])
                                    ]
                                }
                            }
                            
                            print(f"Debug - Formatted question: {formatted_question}")
                            formatted_output["questions"].append(formatted_question)
                            question_counter += 1
                            print(f"Successfully extracted question_{question_counter-1}")
                            
                    except Exception as format_error:
                        print(f"Error formatting question: {format_error}")
                        continue
            else:
                print(f"Warning: Invalid response structure in chunk {i//chunk_size + 1}")
                print("Expected 'questions' key in:", response_data)
            
            print(f"Completed processing chunk {i//chunk_size + 1}")
                
        total_questions = len(formatted_output["questions"])
        print(f"\nTotal questions extracted: {total_questions}")
        if total_questions == 0:
            print("Warning: No questions were extracted!")
            
        return formatted_output  # Return the formatted output instead of raw response

    except Exception as e:
        print(f"Error in extract_questions: {str(e)}")
        print(f"Traceback:\n{traceback.format_exc()}")
        return {"questions": []}

In [9]:
def rephrase_questions(conf_data: dict, all_questions: dict, topics: list) -> dict:
    print("Starting the cleanup and rephrasing process...")
    try:
        # Validate inputs
        if not conf_data or "chapter_name" not in conf_data:
            raise ValueError("Missing chapter_name in configuration")
            
        if not topics:
            print("\n\nNo topics found in the data.")
            raise ValueError("No topics found in the data")
            
        chapter = conf_data["chapter_name"]
        
        # Validate input questions format
        if not isinstance(all_questions, dict) or "questions" not in all_questions:
            raise ValueError("Invalid input questions format")
            
        questions_list = all_questions["questions"]
        if not questions_list:
            raise ValueError(f"No questions found for chapter {chapter}")
            
        print(f"\nWorking on chapter {chapter}")
        print(f"Total questions to process: {len(questions_list)}")
        
        # Initialize to store all batches
        all_rephrased_questions = {
            "questions": [],
            "topic": "",
            "topic_id": ""
        }
        
        # Process questions in batches of 5
        batch_size = 5
        for batch_start in range(0, len(questions_list), batch_size):
            try:
                # Get current batch of questions
                batch_end = min(batch_start + batch_size, len(questions_list))
                question_batch = questions_list[batch_start:batch_end]
                
                print(f"\nProcessing batch {batch_start//batch_size + 1}/{-(-len(questions_list)//batch_size)}")
                print(f"Debug - Processing questions: {question_batch}")
                
                # Clean and rephrase the batch
                response = clean_rephrase_question(question_batch, topics)
                print("Raw response:", response)
                
                # Parse the cleaned response
                if response and hasattr(response, 'content'):
                    # Clean the content string before parsing
                    content = response.content
                    if content.startswith('```json'):
                        content = content[7:]
                    if content.endswith('```'):
                        content = content[:-3]
                        
                    try:
                        formatted_response = parse_Response(content)
                        if formatted_response and isinstance(formatted_response, dict):
                            # Store topic information from first successful batch
                            if not all_rephrased_questions["topic"]:
                                all_rephrased_questions["topic"] = formatted_response.get("topic", "")
                                all_rephrased_questions["topic_id"] = formatted_response.get("topic_id", "")
                                
                            # Add questions from this batch
                            if "questions" in formatted_response:
                                all_rephrased_questions["questions"].extend(formatted_response["questions"])
                                print(f"Successfully processed batch with {len(formatted_response['questions'])} questions")
                                print(f"Topic: {formatted_response.get('topic', 'Unknown')}")
                                print(f"Topic ID: {formatted_response.get('topic_id', 'Unknown')}")
                    except Exception as parse_error:
                        print(f"Error in parse_Response: {str(parse_error)}")
                        print("Content being parsed:", content)
                        continue
                
                print("-" * 100)
                
            except Exception as batch_error:
                print(f"Error processing batch {batch_start//batch_size + 1}: {str(batch_error)}")
                print(f"Debug - Error details: {str(batch_error)}")
                print(f"Debug - Batch content: {question_batch}")
                continue
        
        # Return all processed questions
        if all_rephrased_questions["questions"]:
            print(f"\nSuccessfully processed total {len(all_rephrased_questions['questions'])} questions")
            return all_rephrased_questions
        else:
            print("\nNo questions were successfully processed")
            return {
                "questions": [],
                "topic": "",
                "topic_id": ""
            }
        
    except Exception as e:
        print(f"Failed to clean and rephrase questions. Exception Occurred: {type(e).__name__} – {str(e)}")
        print(f"Traceback:\n{traceback.format_exc()}")
        return {
            "questions": [],
            "topic": "",
            "topic_id": ""
        }

In [10]:
# get chapter and book data from defaultConf
with open("image_conf.json", "r") as f:
    conf_data = json.load(f)

chapter = conf_data["chapter_name"]
subject = conf_data["subject"]


print("Chapter to work on: ", chapter)
print(subject)
print(conf_data['publication'])

Chapter to work on:  Triangles
mathematics
mtg


In [11]:
from typing import Tuple, Dict, Optional
import traceback
import logging

def extract_rephrase_questions(conf_data: dict, topics: list) -> Tuple[Optional[Dict], Optional[Dict]]:
    try:
        # Validate configuration data
        required_fields = ["subject", "grade", "chapter_name"]
        missing_fields = [field for field in required_fields if field not in conf_data]
        if missing_fields:
            raise ValueError(f"Missing required configuration fields: {', '.join(missing_fields)}")
            
        if not topics:
            raise ValueError("No topics provided for question rephrasing")
            
        print("Starting question extraction and rephrasing pipeline...")
        
        # Step 1: Scrape text from PDF/LaTeX
        print("\nStep 1: Scraping textbook content...")
        pdf_text = read_data_from_latex(publication=conf_data['publication'] , chapter_name=conf_data['chapter_name'] ,grade=conf_data['grade'])
        if not pdf_text:
            raise ValueError("No text content extracted from textbook")
        print(f"Successfully extracted {len(pdf_text)} text segments")
            
        # Step 2: Extract questions from text
        print("\nStep 2: Extracting questions from text...")
        extracted_questions = extract_questions(
            subject=conf_data["subject"],
            grade=conf_data["grade"],
            chapter_name=conf_data["chapter_name"],
            chapter_texts=pdf_text.split("\n")
        )
        if not extracted_questions:
            raise ValueError("No questions were extracted from the text")
        print(f"Successfully extracted questions")
            
        # Step 3: Clean and rephrase questions
        print("\nStep 3: Cleaning and rephrasing questions...")
        final_rephrased_questions = rephrase_questions(
            conf_data=conf_data,
            all_questions=extracted_questions,
            topics=topics
        )
        if not final_rephrased_questions:
            raise ValueError("No questions were successfully rephrased")
        print("Successfully rephrased questions")
            
        # Return results
        print("\nPipeline completed successfully!")
        return extracted_questions , final_rephrased_questions
    except ValueError as val_err:
        print(f"Validation error: {str(val_err)}")
        print(f"Traceback:\n{traceback.format_exc()}")
        return None, None
        
    except Exception as e:
        print(f"Unexpected error in question processing pipeline: {str(e)}")
        print(f"Exception type: {type(e).__name__}")
        print(f"Traceback:\n{traceback.format_exc()}")
        return None, None
        
    finally:
        print("\nQuestion processing pipeline finished")

In [12]:
import pandas as pd

def process_filename(filename):
    """
    Process filename by stripping whitespace and converting to lowercase
    """
    return filename.strip().lower()
grade = conf_data['grade']
# Match subject to determine sheet name
match subject:
    case "mathematics":
        sheet_name = f"G{grade} Maths"
    case "biology":
        sheet_name = f"G{grade} Science"
    case "physics":
        sheet_name = f"G{grade} Science"
    case "chemistry":
        sheet_name = f"G{grade} Science"
print(sheet_name)
# Read Excel file and process the filename
file_name = process_filename("LEAP Course Creation - Topic LU Prerequisite Misconceptions.xlsx")
misconceptions_df = pd.read_excel(file_name, sheet_name=sheet_name)

# Clean column names
misconceptions_df.columns = misconceptions_df.columns.str.strip()
misconceptions_df.fillna("", inplace=True)

# Process chapter name from conf_data for comparison
chapter_name = conf_data["chapter_name"].strip().lower()

# Get unique topics for the specified chapter
tempTopics = list(misconceptions_df[
    misconceptions_df["Chapter (NCERT/AcadAlly's name)"].str.strip().str.lower() == chapter_name
]["Topic"].unique())

# Create topics dictionary with processed IDs
topics = {}
for i in range(len(tempTopics)):
    topic_id = f"{conf_data['grade']}_{conf_data['subject']}_{chapter_name}_{i+1}"
    topics[topic_id] = tempTopics[i].strip()

print("Topics with ID: ", topics, "\n\n", topics.values())

# Process misconceptions and LUs
total_misconceptions = {}
total_LUs = {}

for i, row in misconceptions_df.iterrows():
    topic = str(row["Topic"]).strip()
    
    if topic in topics.values():
        # Process LUs
        if topic in total_LUs:
            total_LUs[topic].append(row["LUs Covered"])
        else:
            total_LUs[topic] = [row["LUs Covered"]]

        # # Process misconceptions
        # temp_misconceptions = [
        #     row[f"Misconceptions {i}"] 
        #     for i in range(1, 11) 
        #     if row[f"Misconceptions {i}"] != ""
        # ]

        # if topic in total_misconceptions:
        #     total_misconceptions[topic].extend(temp_misconceptions)
        # else:
        #     total_misconceptions[topic] = temp_misconceptions

print(total_LUs, sep="\n\n-------------------------------------------------------------------\n\n")

G10 Maths
Topics with ID:  {'10_mathematics_triangles_1': 'Similar Figures', '10_mathematics_triangles_2': 'Basic Proportionality Theorem', '10_mathematics_triangles_3': 'Criteria for Similarity of Triangles: AAA', '10_mathematics_triangles_4': 'Criteria for Similarity of Triangles: SSS', '10_mathematics_triangles_5': 'Criteria for Similarity of Triangles: SAS'} 

 dict_values(['Similar Figures', 'Basic Proportionality Theorem', 'Criteria for Similarity of Triangles: AAA', 'Criteria for Similarity of Triangles: SSS', 'Criteria for Similarity of Triangles: SAS'])
{'Similar Figures': ['Criteria for similarity of figures'], 'Basic Proportionality Theorem': ['Basic Proportionality Theorem (Thales Theorem)', 'Converse of Basic Proportionality Theorem (Thales theorem)'], 'Criteria for Similarity of Triangles: AAA': ['Criteria for similarity of triangles: AAA (or AA)'], 'Criteria for Similarity of Triangles: SSS': ['Criteria for similarity of triangles: SSS'], 'Criteria for Similarity of Tria

C:\Users\aniket singh\AppData\Local\Temp\ipykernel_24708\609127329.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  misconceptions_df.fillna("", inplace=True)


In [13]:
total_LUs

{'Similar Figures': ['Criteria for similarity of figures'],
 'Basic Proportionality Theorem': ['Basic Proportionality Theorem (Thales Theorem)',
  'Converse of Basic Proportionality Theorem (Thales theorem)'],
 'Criteria for Similarity of Triangles: AAA': ['Criteria for similarity of triangles: AAA (or AA)'],
 'Criteria for Similarity of Triangles: SSS': ['Criteria for similarity of triangles: SSS'],
 'Criteria for Similarity of Triangles: SAS': ['Criteria for similarity of triangles: SAS']}

In [14]:
# function call for execution
extracted_raw_questions_json , clean_rephrased_questions_json  = extract_rephrase_questions(conf_data, topics)
# clean_rephrased_questions_json

Starting question extraction and rephrasing pipeline...

Step 1: Scraping textbook content...
Successfully processed LaTeX file from zip: C:\Users\ANIKET~1\AppData\Local\Temp\tmpdsm9diqr\2025_02_17_bbe3f767e7e1821afad5g\2025_02_17_bbe3f767e7e1821afad5g.tex
Successfully extracted 98302 text segments

Step 2: Extracting questions from text...

Processing chunk 1/8

Debug - Raw response: content='{\n    "questions": \n    [\n        {\n            "question": \n            {\n                "question_text": "& In a triangle, a line drawn parallel to one side, to intersect the other sides in distinct points, divides the two sides in the same ratio. In $\\\\triangle A B C$, if $D E \\\\| B C$. Then\\\\\\\\",\n                "image_ids": \n                [\n                    {"image_1":"2025_02_17_bbe3f767e7e1821afad5g-324"}\n                ]\n            }\n        },\n        {\n            "question": \n            {\n                "question_text": "& 4. In $\\\\triangle A B C, D 

In [15]:
jv,bhmb

NameError: name 'jv' is not defined

In [16]:
extracted_raw_questions_json

{'questions': [{'question_1': {'question_text': '& In a triangle, a line drawn parallel to one side, to intersect the other sides in distinct points, divides the two sides in the same ratio. In $\\triangle A B C$, if $D E \\| B C$. Then\\\\',
    'image_ids': [{'image_1': '2025_02_17_bbe3f767e7e1821afad5g-324'}]}},
  {'question_2': {'question_text': '& 4. In $\\triangle A B C, D E \\| B C$ (as shown in the figure). If $A D=2 \\mathrm{~cm}, B D=3 \\mathrm{~cm}, B C=7.5 \\mathrm{~cm}$, then the length of $D E$ (in cm ) is:\\\\',
    'image_ids': [{'image_1': '2025_02_17_bbe3f767e7e1821afad5g-325'}]}},
  {'question_3': {'question_text': '& 5. In the given figure, in $\\triangle A B C, D E \\| B C$. If $A D=2.4 \\mathrm{~cm}, D B=4 \\mathrm{~cm}$ and $A E=2 \\mathrm{~cm}$, then the length of $A C$ is\\\\',
    'image_ids': [{'image_1': '2025_02_17_bbe3f767e7e1821afad5g-326(2)'}]}},
  {'question_4': {'question_text': '& 6. In the given figure, $D E \\| B C$ and all measurements are given in

In [17]:
print("Chapter name: ",chapter, ",  Raw questions: ",  len(extracted_raw_questions_json['questions']))
print("Chapter name: ",chapter, ",  Cleaned questions: ",  len(clean_rephrased_questions_json['questions']))

Chapter name:  Triangles ,  Raw questions:  101
Chapter name:  Triangles ,  Cleaned questions:  99


In [18]:
clean_rephrased_questions_json

{'questions': [{'question': {'question_text': '\\begin{aligned} &\\text{In } \\triangle ABC, \\text{ if } DE \\parallel BC, \\text{ what is the relationship between the segments } AD, DB, AE, \\text{ and } EC? \\end{aligned}',
    'image_ids': [{'image_1': '2025_02_17_bbe3f767e7e1821afad5g-324'}]}},
  {'question': {'question_text': '\\begin{aligned} &\\text{In } \\triangle ABC, \\text{ with } DE \\parallel BC, \\text{ given } AD = 2 \\, \\text{cm}, BD = 3 \\, \\text{cm}, \\text{ and } BC = 7.5 \\, \\text{cm}, \\text{ calculate the length of } DE. \\end{aligned}',
    'image_ids': [{'image_1': '2025_02_17_bbe3f767e7e1821afad5g-325'}]}},
  {'question': {'question_text': '\\begin{aligned} &\\text{In the figure, } \\triangle ABC \\text{ has } DE \\parallel BC. \\text{ If } AD = 2.4 \\, \\text{cm}, DB = 4 \\, \\text{cm}, \\text{ and } AE = 2 \\, \\text{cm}, \\text{ find the length of } AC. \\end{aligned}',
    'image_ids': [{'image_1': '2025_02_17_bbe3f767e7e1821afad5g-326(2)'}]}},
  {'ques

# Solution and Distractors(Based on misconceptions)

In [19]:
llm = ChatOpenAI(model="gpt-4o", temperature=0.2, api_key = OPENAI_API_KEY ,  request_timeout=30.0)

In [20]:
questions_df = pd.DataFrame(clean_rephrased_questions_json)
questions_df

,questions,topic,topic_id
0,{'question': {'question_text': '\begin{aligned...,Basic Proportionality Theorem,10_mathematics_triangles_2
1,{'question': {'question_text': '\begin{aligned...,Basic Proportionality Theorem,10_mathematics_triangles_2
2,{'question': {'question_text': '\begin{aligned...,Basic Proportionality Theorem,10_mathematics_triangles_2
3,{'question': {'question_text': '\begin{aligned...,Basic Proportionality Theorem,10_mathematics_triangles_2
4,{'question': {'question_text': '\begin{aligned...,Basic Proportionality Theorem,10_mathematics_triangles_2
...,...,...,...
94,{'question': {'question_text': '\begin{aligned...,Basic Proportionality Theorem,10_mathematics_triangles_2
95,{'question': {'question_text': '\begin{aligned...,Basic Proportionality Theorem,10_mathematics_triangles_2
96,{'question': {'question_text': '\begin{aligned...,Basic Proportionality Theorem,10_mathematics_triangles_2
97,{'question': {'question_text': '\begin{aligned...,Basic Proportionality Theorem,10_mathematics_triangles_2


In [21]:
import json
def parse_Response(response):
    if isinstance(response, str):
        # print(response)
        start_index = response.find("{")
        end_index = response.rfind("}")
        if start_index != -1 and end_index != -1:
            valid_json_content = response[start_index : end_index + 1]
            try:
                JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
                # append_list_to_file(JSON_response)
                return JSON_response
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON response: {e.__class__.__name__} - {e}\n\n Still trying to work on particular exceptions ...")
                if str(e).startswith("Extra data"):
                    json_parts = valid_json_content.split('}\n{')
                    if json_parts:
                        first_json = json_parts[0] + '}'
                        return parse_Response(first_json)  
                elif str(e).startswith("Invalid \escape"):
                    print("Before removing \\ issue: ", valid_json_content)
                    str1 = valid_json_content.replace("\\\\\\\\\\", "fvback")
                    str1 = str1.replace("\\\\\\\\", "frback")
                    str1 = str1.replace("\\\\\\", "trlback")
                    str1 = str1.replace("\\\\", "dblback")
                    str1 = str1.replace("\\", "\\\\")
                    str1 = str1.replace("fvback","\\\\\\\\\\")
                    str1 = str1.replace("frback", "\\\\\\\\")
                    str1 = str1.replace("trlback","\\\\\\")
                    strfinal = str1.replace("dblback","\\\\")
                    print("After removing \\ issue: ", strfinal)
                    JSON_response = json.loads(strfinal.replace("\n", ""))
                    return JSON_response
                else:
                    print("Actual Content: ", valid_json_content)
        else:
            print("No valid JSON content found in the response.")
        # time.sleep(50)
    elif isinstance(response, dict):
        return response
    else:
        print("No response message found", type(response))

<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
<>:10: SyntaxWarning: invalid escape sequence '\('
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:20: SyntaxWarning: invalid escape sequence '\e'
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_24708\3339088456.py:10: SyntaxWarning: invalid escape sequence '\('
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_24708\3339088456.py:10: SyntaxWarning: invalid escape sequence '\)'
  JSON_response = json.loads(valid_json_content.replace("\n", "").replace("\(", "").replace("\)", ""))
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_24708\3339088456.py:20: SyntaxWarning: invalid escape sequence '\e'
  elif str(e).startswith("Invalid \escape"):


In [22]:
prompt_template = PromptTemplate(
    input_variables=["system_prompt", "question", "topic", "topic_id", "lulist"],
    template="{system_prompt}\n\n\nGenerate a hint, and solution for the given question. Also rephrase the given question according to the specified steps.\nHere are the required context:\nQuestion: {question}\nLearning unit data which defines scope from which solution should be generated: {lulist}\nTopic: {topic}\nTopic ID: {topic_id}\n\nReturn the output in the specified JSON format."
)

chain = LLMChain(llm=llm, prompt=prompt_template)

def generate_explanation(subject, question_data, topic, topic_id):
    try:
        match subject:
            case "mathematics":
                system = MATHEMATICS_SOLUTION_SYSTEM_PROMPT
            case "biology":
                system = BIOLOGY_SOLUTION_SYSTEM_PROMPT
            case "physics":
                system = PHYSICS_SOLUTION_SYSTEM_PROMPT
            case "chemistry":
                system = CHEMISTRY_SOLUTION_SYSTEM_PROMPT

        # Construct question text with image references
        question_text = question_data['question_text']
        if 'image_ids' in question_data:
            image_refs = [list(img.values())[0] for img in question_data['image_ids']]
            image_str = "\nReference Images: " + ", ".join(image_refs)
            question_text += image_str

        response = chain.run(
            system_prompt=system,
            lulist=total_LUs,
            question=question_text,
            topic=topic,
            topic_id=topic_id
        )
        return response
    except Exception as e:
        print(f"Error processing question: {question_text}\nError: {str(e)}")
        return None

explanations = []

def process_questions(subject, questions_data):
    questions = questions_data['questions']
    topic = questions_data['topic']
    topic_id = questions_data['topic_id']
    
    counter = 0
    retries_counter = 0
    maxCounter = len(questions)
    
    while counter < maxCounter:
        question_data = questions[counter]['question']
        print(f"Processing question {counter + 1} out of {len(questions)}")
        
        explanation = generate_explanation(subject, question_data, topic, topic_id)
        parsed_explanation = parse_Response(explanation)
        
        if parsed_explanation:
            parsed_explanation["question"] = question_data['question_text']
            parsed_explanation["topic"] = topic
            parsed_explanation["topic_id"] = topic_id
            # Add image references to the parsed explanation if they exist
            if 'image_ids' in question_data:
                parsed_explanation["image_ids"] = question_data['image_ids']
            
            explanations.append(parsed_explanation)
            counter += 1
            retries_counter = 0
        else:
            print("Issue in openai response", explanation, "\n\n Inputs: \n\n", 
                  question_data['question_text'], "\n\n Topic: ", topic)
            print(f"Failed to parse explanation or generate solution for question {counter + 1}, retrying...")
            retries_counter += 1
            
        if retries_counter == 3:
            counter += 1

process_questions(subject, clean_rephrased_questions_json)

C:\Users\aniket singh\AppData\Local\Temp\ipykernel_24708\840063619.py:6: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt_template)
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_24708\840063619.py:27: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = chain.run(


Processing question 1 out of 99
Processing question 2 out of 99
Error decoding JSON response: JSONDecodeError - Invalid \escape: line 1 column 389 (char 388)

 Still trying to work on particular exceptions ...
Before removing \ issue:  {
  "hint": "\\begin{aligned} &\\text{Use the Basic Proportionality Theorem to find the length of } DE. \\end{aligned}",
  "solution": "\\begin{aligned} &\\text{Given } DE \\parallel BC, \\text{ by the Basic Proportionality Theorem:} \\\\ &\\frac{AD}{DB} = \\frac{DE}{BC} \\\\ &\\implies \\frac{2}{3} = \\frac{DE}{7.5} \\\\ &\\therefore DE = \\frac{2}{3} \\times 7.5 \\\\ &\\implies DE = 5 \, \\text{cm} \\\\ \\end{aligned}"
}
After removing \ issue:  {
  "hint": "\\begin{aligned} &\\text{Use the Basic Proportionality Theorem to find the length of } DE. \\end{aligned}",
  "solution": "\\begin{aligned} &\\text{Given } DE \\parallel BC, \\text{ by the Basic Proportionality Theorem:} \\\\ &\\frac{AD}{DB} = \\frac{DE}{BC} \\\\ &\\implies \\frac{2}{3} = \\frac{DE

In [28]:
explanations

[{'hint': '\\begin{aligned} &\\text{Apply the Basic Proportionality Theorem (Thales Theorem) to } \\triangle ABC \\text{ with } DE \\parallel BC \\end{aligned}',
  'solution': '\\begin{aligned} &\\text{Given } DE \\parallel BC \\text{ in } \\triangle ABC \\\\ &\\implies \\frac{AD}{DB} = \\frac{AE}{EC} \\quad \\text{[By Basic Proportionality Theorem]} \\\\ &\\therefore \\text{The relationship is } \\frac{AD}{DB} = \\frac{AE}{EC} \\\\ \\end{aligned}',
  'question': '\\begin{aligned} &\\text{In } \\triangle ABC, \\text{ if } DE \\parallel BC, \\text{ what is the relationship between the segments } AD, DB, AE, \\text{ and } EC? \\end{aligned}',
  'topic': 'Basic Proportionality Theorem',
  'topic_id': '10_mathematics_triangles_2',
  'image_ids': [{'image_1': '2025_02_17_bbe3f767e7e1821afad5g-324'}]},
 {'hint': '\\begin{aligned} &\\text{Use the Basic Proportionality Theorem to find the length of } DE. \\end{aligned}',
  'solution': '\\begin{aligned} &\\text{Given } DE \\parallel BC, \\text{

### Generating Distractors

In [24]:
data = pd.DataFrame(explanations)


In [25]:
data['misconception_options'] = ''
data

,hint,solution,question,topic,topic_id,image_ids,misconception_options
0,\begin{aligned} &\text{Apply the Basic Proport...,\begin{aligned} &\text{Given } DE \parallel BC...,"\begin{aligned} &\text{In } \triangle ABC, \te...",Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,
1,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Given } DE \parallel BC...,"\begin{aligned} &\text{In } \triangle ABC, \te...",Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,
2,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\frac{AD}{DB} = \frac{AE}{EC}...,"\begin{aligned} &\text{In the figure, } \trian...",Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,
3,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Given } DE \parallel BC...,"\begin{aligned} &\text{In the figure, } DE \pa...",Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,
4,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Given } DE \parallel BC...,"\begin{aligned} &\text{In the figure, } DE \pa...",Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,
...,...,...,...,...,...,...,...
94,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Given } PQ \parallel BC...,\begin{aligned} &\text{ If } P \quad Q \mid \m...,Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,
95,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Given } \triangle ABC \...,\begin{aligned} &\text{ In } \triangle A \quad...,Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,
96,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Since } E \quad F \mid ...,\begin{aligned} &\text{ In triangle } A \quad ...,Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,
97,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Given } \angle PQR = 90...,\begin{aligned} &\text{ In } \triangle P \quad...,Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,


In [26]:
for i in range(len(data)):
    print(data['solution'][i].split("\n"))

['\\begin{aligned} &\\text{Given } DE \\parallel BC \\text{ in } \\triangle ABC \\\\ &\\implies \\frac{AD}{DB} = \\frac{AE}{EC} \\quad \\text{[By Basic Proportionality Theorem]} \\\\ &\\therefore \\text{The relationship is } \\frac{AD}{DB} = \\frac{AE}{EC} \\\\ \\end{aligned}']
['\\begin{aligned} &\\text{Given } DE \\parallel BC, \\text{ by the Basic Proportionality Theorem:} \\\\ &\\frac{AD}{DB} = \\frac{DE}{BC} \\\\ &\\implies \\frac{2}{3} = \\frac{DE}{7.5} \\\\ &\\therefore DE = \\frac{2}{3} \\times 7.5 \\\\ &\\implies DE = 5 \\, \\text{cm} \\\\ \\end{aligned}']
['\\begin{aligned} &\\frac{AD}{DB} = \\frac{AE}{EC} \\\\ &\\implies \\quad \\frac{2.4}{4} = \\frac{2}{EC} \\\\ &\\implies \\quad EC = \\frac{2 \\times 4}{2.4} \\\\ &\\implies \\quad EC = \\frac{8}{2.4} \\\\ &\\implies \\quad EC = 3.3333 \\, \\text{cm} \\\\ &\\therefore \\quad AC = AE + EC = 2 + 3.3333 = 5.3333 \\, \\text{cm} \\\\ \\end{aligned}']
['\\begin{aligned} &\\text{Given } DE \\parallel BC, \\text{ by Basic Proportio

In [27]:
# Define the prompt template for generating misconceptions
prompt_template = PromptTemplate(
    input_variables=["system_prompt","topic", "topic_id", "question", "hint", "solution"],
    template="{system_prompt}\n\nGenerate one correct option and appropriate incorrect options in latex using given solution, hint, and misconceptions for the given question.\n\nTopic ID: {topic_id}\nTopic: {topic}\nQuestion: {question}\nHint: {hint}\nSolution: {solution}\n\nReturn the output in the specified JSON format."
)

# Extract relevant information from the 'explanation' column

def extract_explanation_details(explanation):
    try:
        explanation_data = json.loads(explanation)
        hint = explanation_data.get('hint', 'No hint available')
        solution = explanation_data.get('solution', 'No solution available')
        return hint, solution
    except json.JSONDecodeError:
        return 'No hint available', 'No solution available'

# Function to generate misconceptions-based incorrect options

def generate_incorrect_options(row):

    chain = LLMChain(llm=llm, prompt=prompt_template)
    try:
        response = chain.run(
            system_prompt=DISTRACTOR_SYSTEM_PROMPT,
            topic=row['topic'],
            topic_id=row['topic_id'],
            question=row['question'],
            hint=row['hint'],
            solution=row['solution']
        )
        # if response.strip().startswith("```json"):
        #     response = response.strip().strip("```json").strip("```").strip()
        return response
    except Exception as e:
        print(f"Unexpected error: {str(e)}")
        return None

# Apply the function to generate misconceptions-based incorrect options for each question
misconception_options = []
def loopForDistractor(data):
    distractor_counter = 0
    retries_counter = 0
    maxCounter = len(data)
    while distractor_counter<maxCounter:
        row = data.loc[distractor_counter]
        print(f"Processing question {distractor_counter + 1} out of {len(data)}")
        misconception_option = generate_incorrect_options(row)
        parsed_response = parse_Response(misconception_option)

        if parsed_response:
            # misconception_options.append(parsed_response)
            # row['misconception_options'] = parsed_response
            # print(distractor_counter)
            # print(data.loc[distractor_counter , 'misconception_options'])
            # print(parsed_response)

            data.loc[distractor_counter , 'misconception_options'] =[parsed_response]
            distractor_counter += 1
            retries_counter = 0
        else:
            print(f"Failed to parse response or generate distractors for question {distractor_counter + 1}, retrying...")
            retries_counter += 1
        if retries_counter == 3:
            # misconception_option.append("")
            data.loc[distractor_counter , 'misconception_options'] = ""
            distractor_counter += 1

loopForDistractor(data)
# data['misconception_options'] = misconception_options

Processing question 1 out of 99
Processing question 2 out of 99
Processing question 3 out of 99
Processing question 4 out of 99
Processing question 5 out of 99
Processing question 6 out of 99
Processing question 7 out of 99
Unexpected error: Request timed out.
No response message found <class 'NoneType'>
Failed to parse response or generate distractors for question 7, retrying...
Processing question 7 out of 99
Processing question 8 out of 99
Processing question 9 out of 99
Processing question 10 out of 99
Processing question 11 out of 99
Processing question 12 out of 99
Processing question 13 out of 99
Processing question 14 out of 99
Processing question 15 out of 99
Processing question 16 out of 99
Processing question 17 out of 99
Processing question 18 out of 99
Processing question 19 out of 99
Processing question 20 out of 99
Processing question 21 out of 99
Processing question 22 out of 99
Processing question 23 out of 99
Processing question 24 out of 99
Processing question 25 out

In [29]:
for i in range(len(data)):
    print(data['misconception_options'][i])

[{'correct_option': '\\begin{aligned} &\\frac{AD}{DB} = \\frac{AE}{EC} \\end{aligned}', 'option1': {'option': '\\begin{aligned} &\\frac{AD}{AE} = \\frac{DB}{EC} \\end{aligned}', 'rationale': '\\begin{aligned} &\\text{Confused the order of segments in the proportion, reversing the correct relationship} \\end{aligned}'}, 'option2': {'option': '\\begin{aligned} &\\frac{AD}{EC} = \\frac{DB}{AE} \\end{aligned}', 'rationale': '\\begin{aligned} &\\text{Ignored the correct pairing of segments across the parallel lines} \\end{aligned}'}, 'option3': {'option': '\\begin{aligned} &\\frac{AD}{DB} = \\frac{EC}{AE} \\end{aligned}', 'rationale': '\\begin{aligned} &\\text{Forgot to maintain the correct order of segments in the proportion} \\end{aligned}'}}]
[{'correct_option': '\\begin{aligned} &\\text{5 cm} \\end{aligned}', 'option1': {'option': '\\begin{aligned} &\\text{4 cm} \\end{aligned}', 'rationale': '\\begin{aligned} &\\text{Confused the ratio calculation by using incorrect values for } AD \\te

In [30]:
data

,hint,solution,question,topic,topic_id,image_ids,misconception_options
0,\begin{aligned} &\text{Apply the Basic Proport...,\begin{aligned} &\text{Given } DE \parallel BC...,"\begin{aligned} &\text{In } \triangle ABC, \te...",Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,[{'correct_option': '\begin{aligned} &\frac{AD...
1,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Given } DE \parallel BC...,"\begin{aligned} &\text{In } \triangle ABC, \te...",Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,[{'correct_option': '\begin{aligned} &\text{5 ...
2,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\frac{AD}{DB} = \frac{AE}{EC}...,"\begin{aligned} &\text{In the figure, } \trian...",Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,[{'correct_option': '\begin{aligned} &\text{5....
3,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Given } DE \parallel BC...,"\begin{aligned} &\text{In the figure, } DE \pa...",Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,[{'correct_option': '\begin{aligned} &\text{4....
4,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Given } DE \parallel BC...,"\begin{aligned} &\text{In the figure, } DE \pa...",Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,[{'correct_option': '\begin{aligned} &x = \fra...
...,...,...,...,...,...,...,...
94,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Given } PQ \parallel BC...,\begin{aligned} &\text{ If } P \quad Q \mid \m...,Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,[{'correct_option': '\begin{aligned} &\text{Se...
95,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Given } \triangle ABC \...,\begin{aligned} &\text{ In } \triangle A \quad...,Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,[{'correct_option': '\begin{aligned} &\text{Se...
96,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Since } E \quad F \mid ...,\begin{aligned} &\text{ In triangle } A \quad ...,Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,[{'correct_option': '\begin{aligned} &\frac{AE...
97,\begin{aligned} &\text{Use the Basic Proportio...,\begin{aligned} &\text{Given } \angle PQR = 90...,\begin{aligned} &\text{ In } \triangle P \quad...,Basic Proportionality Theorem,10_mathematics_triangles_2,[{'image_1': '2025_02_17_bbe3f767e7e1821afad5g...,[{'correct_option': '\begin{aligned} &\text{Tr...


### Output Formatting & Pushing into the DB

In [31]:
# Function to process each row and extract required fields
def process_row(row, index):
    try:
        misconception_data = row["misconception_options"]
        # Extract topic, question, and solution details
        topic_id = row["topic_id"]
        topic_name = row["topic"]
        question = row["question"]
        hint = row["hint"]
        explanation = row["solution"]
        images = row.get("image_ids", [])
        # correct_answer = row["final_answer"]
        chapter = conf_data['chapter_name']

        # Extract option and rationale details
        options = {}
        for i in range(1, 4):  # Assuming there are 4 options for each question
            options["correct_answer"] = misconception_data[0]["correct_option"]
            option_key = f'option{i}'
            option_info = misconception_data[0][option_key]
            options[f'option{i}'] = option_info.get('option', '')
            options[f'dr{i}'] = option_info.get('rationale', '')
        
        # Construct the final JSON structure
        start = "\\begin{aligned}"
        end = "\\end{aligned}"
        
        combined_data = {
            'topic_id': topic_id,
            'topic_name': topic_name,
            'grade' : conf_data['grade'],
            'board' : "CBSE",
            
            'subject' : subject,
            'chapter_name' : chapter,
            "images": images,
            'publication' : conf_data['publication'],
            'question': [{"content": question}],
            # if (str(question).startswith("\\begin{aligned}") and str(question).endswith("\\end{aligned}")) else [{"content": start + question + end}],

            'hint': [{"content": hint}],
            # if (str(hint).startswith("\\begin{aligned}") and str(hint).endswith("\\end{aligned}")) else [{"content": start + hint + end}],

            'solution': [{"content": explanation}],
            # if (str(explanation).startswith("\\begin{aligned}") and str(explanation).endswith("\\end{aligned}")) else [{"content": start + explanation + end}],

            'final_answer': [{"content": options.get('correct_answer', '')}],
            # if (str(options.get('correct_answer', '')).startswith("\\begin{aligned}") and str(options.get('correct_answer', '')).endswith("\\end{aligned}")) else [{"content": start + options.get('correct_answer', '') + end}],

            'option1': [{"content": options.get('option1', '')}],
            # if (str(options.get('option1')).startswith("\\begin{aligned}") and str(options.get('option1')).endswith("\\end{aligned}")) else [{"content": start + str(options.get('option1', '')) + end} ],
            'dr1': [{"content": options.get('dr1', '')}],
            'option2': [{"content": options.get('option2', '')}],
            # if (str(options.get('option2')).startswith("\\begin{aligned}") and str(options.get('option2')).endswith("\\end{aligned}")) else [{"content": start + str(options.get('option2', '')) + end}],
            'dr2': [{"content": options.get('dr2', '')}],
            'option3': [{"content": options.get('option3', '')}],
            # if (str(options.get('option3')).startswith("\\begin{aligned}") and str(options.get('option3')).endswith("\\end{aligned}")) else [{"content": start + str(options.get('option3', '')) + end}],
            'dr3': [{"content": options.get('dr3', '')}]


        }

        return combined_data
    except Exception as e:
        # If there is an error, return an empty structure with an error message
        print(f"Exception Occurred: {type(e).__name__} – {e} \n {traceback.format_exc()}")

# Process all rows in the dataframe
final_output = [process_row(row, idx) for idx, row in data.iterrows()]


In [43]:
final_output

[{'topic_id': '10_mathematics_triangles_2',
  'topic_name': 'Basic Proportionality Theorem',
  'grade': '10',
  'board': 'CBSE',
  'subject': 'mathematics',
  'chapter_name': 'Triangles',
  'images': [{'image_1': '2025_02_17_bbe3f767e7e1821afad5g-324'}],
  'publication': 'mtg',
  'question': [{'content': '\\begin{aligned} &\\text{In } \\triangle ABC, \\text{ if } DE \\parallel BC, \\text{ what is the relationship between the segments } AD, DB, AE, \\text{ and } EC? \\end{aligned}'}],
  'hint': [{'content': '\\begin{aligned} &\\text{Apply the Basic Proportionality Theorem (Thales Theorem) to } \\triangle ABC \\text{ with } DE \\parallel BC \\end{aligned}'}],
  'solution': [{'content': '\\begin{aligned} &\\text{Given } DE \\parallel BC \\text{ in } \\triangle ABC \\\\ &\\implies \\frac{AD}{DB} = \\frac{AE}{EC} \\quad \\text{[By Basic Proportionality Theorem]} \\\\ &\\therefore \\text{The relationship is } \\frac{AD}{DB} = \\frac{AE}{EC} \\\\ \\end{aligned}'}],
  'final_answer': [{'conten

In [ ]:
from docx import Document
from docx.shared import Pt, Inches
import random
import os
from pathlib import Path
from PIL import Image
import io
from pylatexenc.latex2text import LatexNodes2Text

#######################
# Helper Functions
#######################

def process_latex_text(text):
    if not text:
        return ""
    return LatexNodes2Text().latex_to_text(text)

def process_image_ids(images):
    try:
        if not images or not isinstance(images, list):
            return []
        processed_images = []
        for img_dict in images:
            if isinstance(img_dict, dict):
                image_id = next(iter(img_dict.values()))
                processed_images.append(f"{image_id}.jpg")
        return processed_images
    except Exception as e:
        print(f"Error in process_image_ids: {e}")
        return []

def get_content_text(content_list):
    try:
        if not content_list or not isinstance(content_list, list):
            return ""
        if not content_list[0]:
            return ""
        return process_latex_text(content_list[0].get('content', ''))
    except Exception as e:
        print(f"Error in get_content_text: {e}")
        return ""

def find_image_in_directory(image_filename, image_dir):
    try:
        image_dir_path = Path(image_dir).resolve()
        direct_path = image_dir_path / image_filename
        if direct_path.is_file():
            try:
                with Image.open(direct_path) as img:
                    img.verify()
                return str(direct_path)
            except Exception as e:
                print(f"Image validation failed for {direct_path}: {str(e)}")
                return None
        for file_path in image_dir_path.rglob(image_filename):
            if file_path.is_file():
                try:
                    with Image.open(file_path) as img:
                        img.verify()
                    return str(file_path)
                except Exception:
                    continue
        print(f"Image not found or invalid: {image_filename}")
        return None
    except Exception as e:
        print(f"Error finding image {image_filename}: {e}")
        return None

# --- Table creation code with image insertion ---

def create_docx_table_with_images(final_output, image_dir, output_path="output.docx"):
    doc = Document()
    style = doc.styles['Normal']
    style.font.name = 'Cambria Math'
    style.font.size = Pt(11)
    
    headings = {
        "LU": "lu",  
        "LU ID": "lu",  
        "LO": "lo",  
        "LO ID": "lo",  
        "Question Type": "",
        "Question ID": "",  # Always empty
        "Difficulty level": "",
        "Cognitive Dimension('Bloom's Level')": "",
        "Title (Stem and prompt)": "question"
    }
    
    for i in range(len(final_output)):
        table = doc.add_table(0, 4)
        table.style = 'Table Grid'
        heads_list = list(headings.keys())
        
        for j in range(len(heads_list)):
            row1 = table.add_row().cells
            row1[0].text = heads_list[j]
            try:
                if heads_list[j] == "Question ID":
                    row1[1].text = ""
                elif headings[heads_list[j]]:
                    if heads_list[j] == "Title (Stem and prompt)":
                        cell = row1[1]
                        question_text = get_content_text(final_output[i].get("question", []))
                        cell.text = question_text
                        images = final_output[i].get("images", [])
                        image_files = process_image_ids(images)
                        for img_file in image_files:
                            img_path = find_image_in_directory(img_file, image_dir)
                            if img_path:
                                try:
                                    # Read the image bytes and wrap them in a BytesIO object
                                    with open(img_path, 'rb') as f:
                                        image_bytes = f.read()
                                    if image_bytes:
                                        img_io = io.BytesIO(image_bytes)
                                        p = cell.add_paragraph()
                                        run = p.add_run()
                                        run.add_picture(img_io, width=Inches(4))
                                    else:
                                        print(f"Empty image file: {img_path}")
                                except Exception as e:
                                    print(f"Error adding picture from {img_path}: {str(e)}")
                    elif heads_list[j] == "LU":
                        row1[1].text = str(final_output[i].get("lu", ""))
                    elif heads_list[j] == "LU ID":
                        row1[1].text = str(final_output[i].get("lu", ""))
                    elif heads_list[j] == "LO":
                        row1[1].text = str(final_output[i].get("lo", ""))
                    elif heads_list[j] == "LO ID":
                        row1[1].text = str(final_output[i].get("lo", ""))
                    else:
                        row1[1].text = str(final_output[i].get(headings[heads_list[j]], ""))
                else:
                    row1[1].text = ""
            except Exception as e:
                print(f"Error processing row {j}: {e}")
                row1[1].text = ""
            row1[2].text = ""
            row1[3].text = ""
            try:
                row1[1].merge(row1[2]).merge(row1[3])
            except Exception as merge_e:
                print(f"Error merging cells in row {j}: {merge_e}")
        
        # Process Options row
        row2 = table.add_row().cells
        row2[0].text = "Options"
        row2[1].text = ""
        row2[2].text = "Type of Distractor"
        row2[3].text = "Distractor Rationale"
        
        rand = random.randrange(0, 4)
        xx = {
            1: "option1",
            3: "option2",
            5: "option3",
            2: "dr1",
            4: "dr2",
            6: "dr3",
        }
        ss = 1
        for j in range(4):
            row3 = table.add_row().cells
            row3[0].text = chr(65 + j)
            if j == rand:
                row3[1].text = get_content_text(final_output[i].get("final_answer", []))
                row3[2].text = ""
                row3[3].text = "Correct Response"
            else:
                try:
                    row3[1].text = get_content_text(final_output[i].get(xx[ss], []))
                    ss += 1
                    row3[2].text = ""
                    row3[3].text = get_content_text(final_output[i].get(xx[ss], []))
                    ss += 1
                except Exception as e:
                    print(f"Error processing option row {j}: {e}")
                    row3[1].text = ""
                    row3[2].text = ""
                    row3[3].text = ""
        
        # Process Hint row
        row4 = table.add_row().cells
        row4[0].text = "Hint"
        try:
            row4[1].text = get_content_text(final_output[i].get("hint", []))
        except Exception as e:
            print(f"Error processing hint: {e}")
            row4[1].text = ""
        row4[2].text = ""
        row4[3].text = ""
        try:
            row4[1].merge(row4[2]).merge(row4[3])
        except Exception as e:
            print(f"Error merging hint cells: {e}")
        
        # Process Explanation row
        row5 = table.add_row().cells
        row5[0].text = "Explanation"
        try:
            row5[1].text = get_content_text(final_output[i].get("solution", []))
        except Exception as e:
            print(f"Error processing explanation: {e}")
            row5[1].text = ""
        row5[2].text = ""
        row5[3].text = ""
        try:
            row5[1].merge(row5[2]).merge(row5[3])
        except Exception as e:
            print(f"Error merging explanation cells: {e}")
        
        doc.add_page_break()
    
    doc.save(output_path)
    print(f"Document saved at {output_path}")

#######################
# Example Usage
#######################

if __name__ == "__main__":
    # Use a raw string to avoid escape sequence issues.
    image_dir = r"textbooks\\mtg\\Triangles_10\\2025_02_17_bbe3f767e7e1821afad5g\\images"
    
    # final_output should be defined as a list of question dictionaries.
    # For example:
    # final_output = [{...}, {...}, ...]
    
    output_filename = "questions_with_images.docx"
    create_docx_table_with_images(final_output, image_dir, output_path=output_filename)


Error adding picture from E:\Questions_extraction\cbse\textbooks\mtg\Triangles_10\2025_02_17_bbe3f767e7e1821afad5g\images\2025_02_17_bbe3f767e7e1821afad5g-324.jpg: 
Error adding picture from E:\Questions_extraction\cbse\textbooks\mtg\Triangles_10\2025_02_17_bbe3f767e7e1821afad5g\images\2025_02_17_bbe3f767e7e1821afad5g-325.jpg: 
Error adding picture from E:\Questions_extraction\cbse\textbooks\mtg\Triangles_10\2025_02_17_bbe3f767e7e1821afad5g\images\2025_02_17_bbe3f767e7e1821afad5g-326(2).jpg: 
Error adding picture from E:\Questions_extraction\cbse\textbooks\mtg\Triangles_10\2025_02_17_bbe3f767e7e1821afad5g\images\2025_02_17_bbe3f767e7e1821afad5g-326(1).jpg: 
Error adding picture from E:\Questions_extraction\cbse\textbooks\mtg\Triangles_10\2025_02_17_bbe3f767e7e1821afad5g\images\2025_02_17_bbe3f767e7e1821afad5g-326.jpg: 
Error adding picture from E:\Questions_extraction\cbse\textbooks\mtg\Triangles_10\2025_02_17_bbe3f767e7e1821afad5g\images\2025_02_17_bbe3f767e7e1821afad5g-327(2).jpg: 
E

In [32]:
final_output

[{'topic_id': '10_mathematics_triangles_2',
  'topic_name': 'Basic Proportionality Theorem',
  'grade': '10',
  'board': 'CBSE',
  'subject': 'mathematics',
  'chapter_name': 'Triangles',
  'images': [{'image_1': '2025_02_17_bbe3f767e7e1821afad5g-324'}],
  'publication': 'mtg',
  'question': [{'content': '\\begin{aligned} &\\text{In } \\triangle ABC, \\text{ if } DE \\parallel BC, \\text{ what is the relationship between the segments } AD, DB, AE, \\text{ and } EC? \\end{aligned}'}],
  'hint': [{'content': '\\begin{aligned} &\\text{Apply the Basic Proportionality Theorem (Thales Theorem) to } \\triangle ABC \\text{ with } DE \\parallel BC \\end{aligned}'}],
  'solution': [{'content': '\\begin{aligned} &\\text{Given } DE \\parallel BC \\text{ in } \\triangle ABC \\\\ &\\implies \\frac{AD}{DB} = \\frac{AE}{EC} \\quad \\text{[By Basic Proportionality Theorem]} \\\\ &\\therefore \\text{The relationship is } \\frac{AD}{DB} = \\frac{AE}{EC} \\\\ \\end{aligned}'}],
  'final_answer': [{'conten

In [34]:
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
import pandas as pd
import os
from pathlib import Path
from PIL import Image
import io

from pylatexenc.latex2text import LatexNodes2Text

def process_latex_text(text):
    """Convert LaTeX formatted text to plain text using pylatexenc."""
    if not text:
        return ""
    return LatexNodes2Text().latex_to_text(text)


def process_image_ids(images):
    """Process image IDs and add .jpg extension"""
    try:
        if not images or not isinstance(images, list):
            return []
        
        processed_images = []
        for img_dict in images:
            if isinstance(img_dict, dict):
                image_id = next(iter(img_dict.values()))
                processed_images.append(f"{image_id}.jpg")
        
        return processed_images
    except Exception as e:
        print(f"Error in process_image_ids: {e}")
        return []

def get_content_text(content_list):
    """Extract content text from list of dictionaries"""
    try:
        if not content_list or not isinstance(content_list, list):
            return ""
        if not content_list[0]:
            return ""
        return process_latex_text(content_list[0].get('content', ''))
    except Exception as e:
        print(f"Error in get_content_text: {e}")
        return ""

def find_image_in_directory(image_filename, image_dir):
    """Find an image in the directory and its subdirectories using Path"""
    try:
        # Convert to Path object
        image_dir_path = Path(image_dir).resolve()
        
        # First try the direct path in the images directory
        direct_path = image_dir_path / image_filename
        if direct_path.is_file():
            try:
                # Validate image
                with Image.open(direct_path) as img:
                    img.verify()
                return str(direct_path)
            except Exception as e:
                print(f"Image validation failed for {direct_path}: {str(e)}")
                return None
        
        # If not found, search recursively
        for file_path in image_dir_path.rglob(image_filename):
            if file_path.is_file():
                try:
                    with Image.open(file_path) as img:
                        img.verify()
                    return str(file_path)
                except Exception:
                    continue
                
        print(f"Image not found or invalid: {image_filename}")
        return None
    except Exception as e:
        print(f"Error finding image {image_filename}: {e}")
        return None

def add_formatted_paragraph(doc, label, content, bold_label=True, spacing_before=0, spacing_after=0):
    """Add a formatted paragraph with label and content"""
    paragraph = doc.add_paragraph()
    if bold_label and label:
        paragraph.add_run(f"{label}: ").bold = True
    paragraph.add_run(content)
    paragraph.space_before = Pt(spacing_before)
    paragraph.space_after = Pt(spacing_after)
    return paragraph

def process_question_data(doc, question_data):
    """Process a single question's data and add it to the document"""
    try:
        # Add metadata section
        metadata_heading = doc.add_heading('Question Information', level=1)
        metadata_heading.style.font.size = Pt(14)
        
        # Handle metadata fields safely
        metadata_fields = {
            "Topic Name": str(question_data.get('topic_name', '')),
            "Subject": str(question_data.get('subject', '')).title(),
            "Grade": str(question_data.get('grade', '')),
            "Board": str(question_data.get('board', '')),
            "Chapter": str(question_data.get('chapter_name', '')),
            "Publication": str(question_data.get('publication', '')),
            "Topic ID": str(question_data.get('topic_id', ''))
        }
        
        for label, content in metadata_fields.items():
            add_formatted_paragraph(doc, label, content)
        
        # Add question section
        doc.add_heading('Question', level=1)
        question_text = get_content_text(question_data.get('question', []))
        add_formatted_paragraph(doc, "", question_text, bold_label=False, spacing_after=10)
        
        # Add images
        images = question_data.get('images', [])
        if images and isinstance(images, list):
            image_files = process_image_ids(images)
            for img_file in image_files:
                img_path = find_image_in_directory(img_file, image_dir)
                if img_path:
                    try:
                        print(f"Adding image from path: {img_path}")
                        with Image.open(img_path) as img:
                            if img.mode not in ('RGB', 'RGBA'):
                                img = img.convert('RGB')
                            img_byte_arr = io.BytesIO()
                            img.save(img_byte_arr, format='PNG')
                            img_byte_arr.seek(0)
                            picture = doc.add_picture(img_byte_arr, width=Inches(4))
                            last_paragraph = doc.paragraphs[-1]
                            last_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
                    except Exception as e:
                        print(f"Error adding image {img_file}: {str(e)}")
        
        # Add options section
        doc.add_heading('Options', level=2)
        for i in range(1, 4):
            option_text = get_content_text(question_data.get(f'option{i}', []))
            explanation = get_content_text(question_data.get(f'dr{i}', []))
            if option_text:
                add_formatted_paragraph(doc, f"Option {i}", option_text, spacing_before=5)
            if explanation:
                add_formatted_paragraph(doc, "Explanation", explanation, spacing_after=10)
        
        # Add hint section
        hint_text = get_content_text(question_data.get('hint', []))
        if hint_text:
            doc.add_heading('Hint', level=2)
            add_formatted_paragraph(doc, "", hint_text, bold_label=False)
        
        # Add solution section
        solution_text = get_content_text(question_data.get('solution', []))
        if solution_text:
            doc.add_heading('Solution', level=2)
            add_formatted_paragraph(doc, "", solution_text, bold_label=False)
        
        # Add final answer section
        final_answer = get_content_text(question_data.get('final_answer', []))
        if final_answer:
            doc.add_heading('Final Answer', level=2)
            add_formatted_paragraph(doc, "", final_answer, bold_label=False)
        
        # Add page break
        doc.add_page_break()
        
    except Exception as e:
        print(f"Error processing question: {str(e)}")

def create_docx_with_images(json_data, image_dir, output_path='questions_with_images.docx'):
    """Create Word document from JSON data with images"""
    try:
        print("\nProcessing data and creating Word document...")
        
        # Convert image_dir to absolute Path
        image_dir = Path(image_dir).resolve()
        print(f"Using image directory: {image_dir}")
        
        # Validate image directory
        if not image_dir.exists():
            raise ValueError(f"Image directory does not exist: {image_dir}")
        
        # Convert string to list if needed
        if isinstance(json_data, str):
            data = eval(json_data)
        else:
            data = json_data
        
        # Ensure data is a list
        if not isinstance(data, list):
            data = [data]
            
        # Create document
        doc = Document()
        
        # Add title
        title = doc.add_heading('Question Paper', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        # Process each question in the list
        for question_data in data:
            process_question_data(doc, question_data)
        
        # Save the document
        doc.save(output_path)
        print(f"Word document created successfully at: {output_path}")
        return True
        
    except Exception as e:
        print(f"Error creating Word document: {str(e)}")
        return False
# Example usage
if __name__ == "__main__":

    image_dir = Path("textbooks\mtg\Triangles_10\\2025_02_17_bbe3f767e7e1821afad5g\images")
    
    create_docx_with_images(final_output, image_dir , output_path=f"{conf_data['chapter_name']}_{conf_data['publication']}_questions_with images.docx")

<>:218: SyntaxWarning: invalid escape sequence '\m'
<>:218: SyntaxWarning: invalid escape sequence '\m'
C:\Users\aniket singh\AppData\Local\Temp\ipykernel_24708\3976394110.py:218: SyntaxWarning: invalid escape sequence '\m'
  image_dir = Path("textbooks\mtg\Triangles_10\\2025_02_17_bbe3f767e7e1821afad5g\images")



Processing data and creating Word document...
Using image directory: E:\Questions_extraction\cbse\textbooks\mtg\Triangles_10\2025_02_17_bbe3f767e7e1821afad5g\images
Adding image from path: E:\Questions_extraction\cbse\textbooks\mtg\Triangles_10\2025_02_17_bbe3f767e7e1821afad5g\images\2025_02_17_bbe3f767e7e1821afad5g-324.jpg
Adding image from path: E:\Questions_extraction\cbse\textbooks\mtg\Triangles_10\2025_02_17_bbe3f767e7e1821afad5g\images\2025_02_17_bbe3f767e7e1821afad5g-325.jpg
Adding image from path: E:\Questions_extraction\cbse\textbooks\mtg\Triangles_10\2025_02_17_bbe3f767e7e1821afad5g\images\2025_02_17_bbe3f767e7e1821afad5g-326(2).jpg
Adding image from path: E:\Questions_extraction\cbse\textbooks\mtg\Triangles_10\2025_02_17_bbe3f767e7e1821afad5g\images\2025_02_17_bbe3f767e7e1821afad5g-326(1).jpg
Adding image from path: E:\Questions_extraction\cbse\textbooks\mtg\Triangles_10\2025_02_17_bbe3f767e7e1821afad5g\images\2025_02_17_bbe3f767e7e1821afad5g-326.jpg
Adding image from path:

#### Latex Formatting to store in DB

In [ ]:
from config import MONGO_URL , DB_NAME

In [ ]:
dbCollection = "LatexTest"
client = AsyncIOMotorClient(MONGO_URL)

KeyboardInterrupt: 

In [ ]:
async def pushToDB(dbContent, subject):
    try:
        db = client[DB_NAME]
        que_collection = db[dbCollection]

        dbContent["status"] = "not_reviewed"
        dbContent["comment"] = ""
        dbContent["testFlag"] = "true"
        dbContent["subject"] = subject
        await que_collection.insert_one(dbContent)
        print("Data Uploaded to the database successfully.")
    except Exception as e:
        print("Exception Occurred: ", type(e).__name__, "–", e, "\n", traceback.format_exc())

In [ ]:
for doc in final_output:
    await pushToDB(doc, subject)
client.close()

Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
Data Uploaded to the database successfully.
